# Model Registry Tutorial`api.models` is a facade over Datamint's MLflow-backed model registry. It wraps MLflow's`RegisteredModel` / `ModelVersion` objects in plain Python objects (`Model`, `ModelVersion`),so you can register, list, and inspect models without learning MLflow's object model.This notebook covers:- Registering a model with `api.models.create()`- Listing and finding models with `api.models.get_list()` / `get_by_name()`- Filtering to deployed models with `only_deployed=True`- Inspecting a model's versions with `model.get_versions()` / `get_latest_version()`- Reading what a version was trained for: `get_task_type()`, `get_supported_modes()`, `get_annotation_specs()`- Reading training metrics with `get_metrics()`- Checking deployment status with `is_deployed()`

## SetupUpdate `datamint` first if needed, then configure your API key.

In [ ]:
%pip install -U datamint --quiet

In [ ]:
from datamint import Api

api = Api()

## Register A ModelRegistering ahead of training gives you a stable name to reference later. `create()` returns theexisting model instead of raising when one with this name is already registered(`exists_ok=True` by default).

In [ ]:
MODEL_NAME = "tutorial_model_registry_demo"

model = api.models.create(MODEL_NAME, description="Model created for the model registry tutorial")
model.name, model.description

## List And Find Models`get_list()` returns every registered model; `get_by_name()` returns a single one, or `None` if itdoesn't exist.

In [ ]:
all_models = api.models.get_list()
print(f"Registered models: {[m.name for m in all_models]}")

found = api.models.get_by_name(MODEL_NAME)
missing = api.models.get_by_name("does-not-exist")
found.name, missing

## Filter To Deployed ModelsPass `only_deployed=True` to skip models that don't have a deployed image yet. See`05_deployment/01_deploy_registered_model.ipynb` for how to deploy one.

In [ ]:
deployed_models = api.models.get_list(only_deployed=True)
print(f"Deployed models: {[m.name for m in deployed_models]}")

model.is_deployed()

## Inspect Versions Of A Trained ModelThe rest of this notebook needs a model with at least one version behind it, typically oneregistered by a Datamint trainer (see `06_end_to_end`) or by`04_experiment_tracking/01_mlflow_manual_logging.ipynb`. Replace `MODEL_NAME` below with oneyou've already trained.

In [ ]:
MODEL_NAME = "FracAtlas_adapted"  # replace with a model you've already registered/trained

model = api.models.get_by_name(MODEL_NAME)
if model is None:
    raise ValueError(f"Model '{MODEL_NAME}' was not found. Train one first, e.g. via the 06_end_to_end notebooks.")

versions = model.get_versions()
print(f"{MODEL_NAME} has {len(versions)} version(s)")

latest = model.get_latest_version()
latest.version, latest.run_id

## What A Version Was Trained ForIf the version was logged with the `datamint` MLflow flavor (true for anything trained through aDatamint trainer), you can read the task type, supported prediction modes, and annotation specsstraight from the model artifact, without inspecting the training run.

In [ ]:
print("Task type:", latest.get_task_type())
print("Supported modes:", latest.get_supported_modes())
print("Annotation specs:", latest.get_annotation_specs())

# Model.get_supported_modes() is a shortcut that delegates to the latest version
model.get_supported_modes() == latest.get_supported_modes()

## Training Metrics`get_metrics()` reads the metrics logged for this version's training run. It returns `{}` insteadof raising when there's no training run behind the version, for example a model registered fromoutside Datamint.

In [ ]:
latest.get_metrics()

In [ ]:
model.get_metrics()  # shortcut, uses the latest version

## Next StepsOnce you have a model version you're happy with, alias it and deploy it, see`05_deployment/01_deploy_registered_model.ipynb`. Registered models are also createdautomatically when you pass `--ai-model <name>` to `datamint-upload` with a name that doesn'texist yet.